# Cross-Dataset Direction Transfer (CREMA-D)

This notebook tests the strongest novelty claim: **do emotion directions learned from RAVDESS transfer to an entirely different dataset without any retraining?**

We download CREMA-D (~7,400 clips from 91 actors), extract embeddings using our frozen fine-tuned wav2vec2, and then classify using only the RAVDESS-derived direction vectors. If the directions transfer, it suggests they capture universal emotion structure in wav2vec2's representation, not dataset-specific artifacts.

**CREMA-D** (Crowd-sourced Emotional Multimodal Actors Dataset) uses different actors, recording conditions, and sentences than RAVDESS — making it a rigorous out-of-distribution test.

In [ ]:
from __future__ import annotations
import importlib.util, os, shutil, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def running_in_colab():
    try: import google.colab; return True
    except ImportError: return False
IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)

def run_command(cmd, cwd=None):
    print("+", " ".join(cmd)); subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

def ensure_packages():
    pkgs = {"yaml":"pyyaml","pandas":"pandas","numpy":"numpy","matplotlib":"matplotlib",
            "seaborn":"seaborn","torch":"torch","torchaudio":"torchaudio","transformers":"transformers",
            "sklearn":"scikit-learn","tqdm":"tqdm","datasets":"datasets"}
    missing = sorted({p for m,p in pkgs.items() if importlib.util.find_spec(m) is None})
    if missing: run_command([sys.executable,"-m","pip","install","-q",*missing])
    else: print("Required packages available.")

def find_local_project_root():
    cwd = Path.cwd().resolve()
    for c in [cwd, cwd.parent, cwd/"FinalProject"]:
        c = c.resolve()
        if (c/"configs"/"wav2vec.yaml").exists() and (c/"src").exists(): return c
    raise FileNotFoundError("Could not find project root.")

def clone_clean(cr):
    if cr.exists(): shutil.rmtree(cr)
    run_command(["git","clone","--depth","1",REPO_URL,str(cr)]); return cr

def prepare_roots():
    rt=Path("/content")/REPO_NAME; cl=Path("/content")/f"{REPO_NAME}-clean"
    if rt.exists() and (rt/".git").exists():
        try: run_command(["git","-C",str(rt),"fetch","origin"])
        except: pass
        st=subprocess.run(["git","-C",str(rt),"status","--short"],text=True,capture_output=True,check=True).stdout.splitlines()
        ab=subprocess.run(["git","-C",str(rt),"rev-list","--left-right","--count","HEAD...origin/main"],text=True,capture_output=True,check=False)
        a,b=(0,0) if ab.returncode!=0 or not ab.stdout.strip() else map(int,ab.stdout.strip().split())
        if not [l for l in st if l.strip()] and a==0:
            if b>0:
                try: run_command(["git","-C",str(rt),"pull","--ff-only","origin","main"]); cr=rt
                except: cr=clone_clean(cl)
            else: cr=rt
        else: cr=clone_clean(cl)
        return rt,cr
    elif rt.exists(): return rt,clone_clean(cl)
    else: run_command(["git","clone","--depth","1",REPO_URL,str(rt)]); return rt,rt

ensure_packages()
if IS_COLAB:
    from google.colab import drive; drive.mount('/content/drive',force_remount=False)
    PROJECT_ROOT,CODE_ROOT=prepare_roots()
else:
    PROJECT_ROOT=find_local_project_root(); CODE_ROOT=PROJECT_ROOT

os.chdir(PROJECT_ROOT)
for r in [str(CODE_ROOT),str(PROJECT_ROOT)]:
    while r in sys.path: sys.path.remove(r)
sys.path.insert(0,str(CODE_ROOT))
if str(PROJECT_ROOT)!=str(CODE_ROOT): sys.path.insert(1,str(PROJECT_ROOT))
for n in [n for n in sys.modules if n=="src" or n.startswith("src.")]: del sys.modules[n]
print("Project root:",PROJECT_ROOT); print("Code root:",CODE_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
local_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / experiment_name
local_analysis_dir = PROJECT_ROOT / 'artifacts' / 'analysis' / experiment_name
local_output_dir = local_analysis_dir / 'cross_dataset_transfer'
local_output_dir.mkdir(parents=True, exist_ok=True)

drive_run_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / experiment_name if IS_COLAB else None
drive_checkpoint_dir = drive_run_root / 'checkpoint' if drive_run_root else None
drive_analysis_dir = drive_run_root / 'analysis' if drive_run_root else None
drive_output_dir = drive_analysis_dir / 'cross_dataset_transfer' if drive_analysis_dir else None

checkpoint_dir = local_checkpoint_dir
if not (checkpoint_dir / 'model_state.pt').exists() and drive_checkpoint_dir and (drive_checkpoint_dir / 'model_state.pt').exists():
    checkpoint_dir = drive_checkpoint_dir
analysis_source_dir = local_analysis_dir
if not (analysis_source_dir / 'embedding_arrays.npz').exists() and drive_analysis_dir and (drive_analysis_dir / 'embedding_arrays.npz').exists():
    analysis_source_dir = drive_analysis_dir

print('Checkpoint:', checkpoint_dir)
print('Analysis:', analysis_source_dir)
print('Output:', local_output_dir)

## Load RAVDESS Directions

First load the RAVDESS-derived emotion directions that we will transfer.

In [ ]:
from src.analysis.emotion_vectors import (
    build_direction_vectors, compute_class_centroids, load_embedding_artifacts,
    linear_classifier_probabilities, cosine_centroid_predict,
)
from src.analysis.extract_embeddings import load_trained_checkpoint
from src.analysis.advanced_analysis import (
    build_cremad_metadata, direction_classify, evaluate_transfer_directions, CREMAD_EMOTION_MAP,
)

# Load RAVDESS artifacts and build directions
artifacts = load_embedding_artifacts(analysis_source_dir)
label_names = list(artifacts.summary['label_names'])
reference_label = 'neutral'
reference_idx = label_names.index(reference_label)
label_ids = artifacts.true_label_ids
train_mask = artifacts.metadata['split'].to_numpy() == 'train'

final_layer_idx = int(artifacts.layer_embeddings.shape[1] - 1)
ravdess_embeddings = artifacts.layer_embeddings[:, final_layer_idx]

ravdess_centroids = compute_class_centroids(ravdess_embeddings[train_mask], label_ids[train_mask], len(label_names))
ravdess_directions = build_direction_vectors(ravdess_centroids, reference_idx)

# Load model checkpoint
checkpoint = load_trained_checkpoint(checkpoint_dir)
cw = checkpoint.model.classifier.weight.detach().cpu().numpy()
cb = checkpoint.model.classifier.bias.detach().cpu().numpy()

print(f'RAVDESS label order: {label_names}')
print(f'CREMA-D emotion mapping: {CREMAD_EMOTION_MAP}')

## Download and Process CREMA-D

We use the HuggingFace `datasets` library to download CREMA-D audio.

In [ ]:
import torch
import torchaudio
from tqdm import tqdm
from datasets import load_dataset

# Load CREMA-D from HuggingFace
print('Loading CREMA-D dataset from HuggingFace...')
cremad_ds = load_dataset('qmeeus/CREMA-D', split='train', trust_remote_code=True)
print(f'Total CREMA-D samples: {len(cremad_ds)}')

# Build metadata
cremad_files = [sample['file'] if 'file' in sample else sample.get('path', f'sample_{i}')
                for i, sample in enumerate(cremad_ds)]
cremad_metadata = build_cremad_metadata(cremad_files)

# Filter to emotions that overlap with RAVDESS labels
valid_emotions = set(label_names)
cremad_metadata = cremad_metadata[cremad_metadata['final_label'].isin(valid_emotions)].reset_index(drop=True)
cremad_label_ids = np.array([label_names.index(e) for e in cremad_metadata['final_label']])

print(f'\nCREMA-D samples with matching emotions: {len(cremad_metadata)}')
display(cremad_metadata['final_label'].value_counts())

## Extract CREMA-D Embeddings

Run each CREMA-D audio clip through our frozen fine-tuned wav2vec2 to get pooled layer-12 embeddings.

In [ ]:
from transformers import AutoFeatureExtractor
from src.training.train_wav2vec import get_best_available_device

device = get_best_available_device()
model = checkpoint.model.to(device).eval()
feature_extractor = checkpoint.feature_extractor
target_sr = 16000

# Build a filename -> dataset index mapping
file_to_ds_idx = {}
for i, sample in enumerate(cremad_ds):
    fname = sample.get('file', sample.get('path', f'sample_{i}'))
    file_to_ds_idx[fname] = i

# Extract embeddings in batches
cremad_embeddings_list = []
cremad_logits_list = []
batch_size = 16

valid_indices = []
for idx, row in tqdm(cremad_metadata.iterrows(), total=len(cremad_metadata), desc='Extracting CREMA-D embeddings'):
    ds_idx = file_to_ds_idx.get(row['file_name'])
    if ds_idx is None:
        continue
    valid_indices.append(idx)

    sample = cremad_ds[ds_idx]
    audio = sample['audio']
    waveform = np.array(audio['array'], dtype=np.float32)
    sr = audio['sampling_rate']

    # Resample if needed
    if sr != target_sr:
        waveform_tensor = torch.tensor(waveform).unsqueeze(0)
        waveform_tensor = torchaudio.functional.resample(waveform_tensor, sr, target_sr)
        waveform = waveform_tensor.squeeze(0).numpy()

    inputs = feature_extractor(waveform, sampling_rate=target_sr, return_tensors='pt', padding=True)
    input_values = inputs['input_values'].to(device)
    attention_mask = inputs.get('attention_mask', None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)

    with torch.inference_mode():
        outputs = model(input_values=input_values, attention_mask=attention_mask, output_hidden_states=True)
        # Get final layer pooled embedding
        final_hidden = outputs.hidden_states[-1]
        pooled = model.pool_hidden_states(final_hidden, attention_mask)
        cremad_embeddings_list.append(pooled.cpu().numpy())
        cremad_logits_list.append(outputs.logits.cpu().numpy())

cremad_embeddings = np.concatenate(cremad_embeddings_list, axis=0)
cremad_logits = np.concatenate(cremad_logits_list, axis=0)
cremad_metadata = cremad_metadata.loc[valid_indices].reset_index(drop=True)
cremad_label_ids = np.array([label_names.index(e) for e in cremad_metadata['final_label']])

print(f'\nExtracted {cremad_embeddings.shape[0]} embeddings, shape: {cremad_embeddings.shape}')
np.savez_compressed(
    local_output_dir / 'cremad_embeddings.npz',
    embeddings=cremad_embeddings,
    logits=cremad_logits,
    label_ids=cremad_label_ids,
)
cremad_metadata.to_csv(local_output_dir / 'cremad_metadata.csv', index=False)

## Transfer Evaluation

Three classifiers applied to CREMA-D embeddings — all trained only on RAVDESS:
1. **Trained head** — the learned linear classifier (direct transfer)
2. **Centroid classifier** — nearest RAVDESS centroid
3. **Direction-only** — RAVDESS direction projections

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# 1. Trained head
trained_probs = linear_classifier_probabilities(cremad_embeddings, cw, cb)
trained_preds = trained_probs.argmax(axis=1)

# 2. Centroid classifier
centroid_preds = cosine_centroid_predict(cremad_embeddings, ravdess_centroids)

# 3. Direction-only classifier
direction_preds = direction_classify(cremad_embeddings, ravdess_directions, reference_idx)

transfer_rows = []
for method, preds in [('trained_head', trained_preds), ('centroid', centroid_preds), ('direction_only', direction_preds)]:
    transfer_rows.append({
        'method': method,
        'dataset': 'CREMA-D',
        'accuracy': float(accuracy_score(cremad_label_ids, preds)),
        'macro_f1': float(f1_score(cremad_label_ids, preds, average='macro')),
        'weighted_f1': float(f1_score(cremad_label_ids, preds, average='weighted')),
    })

# Add RAVDESS test baselines for comparison
ravdess_test_mask = artifacts.metadata['split'].to_numpy() == 'test'
ravdess_test_ids = label_ids[ravdess_test_mask]
ravdess_test_preds = artifacts.pred_label_ids[ravdess_test_mask]
ravdess_dir_preds = direction_classify(ravdess_embeddings[ravdess_test_mask], ravdess_directions, reference_idx)

transfer_rows.append({
    'method': 'trained_head', 'dataset': 'RAVDESS (test)',
    'accuracy': float(accuracy_score(ravdess_test_ids, ravdess_test_preds)),
    'macro_f1': float(f1_score(ravdess_test_ids, ravdess_test_preds, average='macro')),
    'weighted_f1': float(f1_score(ravdess_test_ids, ravdess_test_preds, average='weighted')),
})
transfer_rows.append({
    'method': 'direction_only', 'dataset': 'RAVDESS (test)',
    'accuracy': float(accuracy_score(ravdess_test_ids, ravdess_dir_preds)),
    'macro_f1': float(f1_score(ravdess_test_ids, ravdess_dir_preds, average='macro')),
    'weighted_f1': float(f1_score(ravdess_test_ids, ravdess_dir_preds, average='weighted')),
})

transfer_df = pd.DataFrame(transfer_rows)
transfer_df.to_csv(local_output_dir / 'transfer_comparison.csv', index=False)
display(transfer_df.round(4))

In [ ]:
# Confusion matrices for CREMA-D
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, preds) in zip(axes, [
    ('Trained Head', trained_preds),
    ('Centroid', centroid_preds),
    ('Direction-Only', direction_preds),
]):
    cm = confusion_matrix(cremad_label_ids, preds, labels=list(range(len(label_names))))
    disp = ConfusionMatrixDisplay(cm, display_labels=label_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    acc = accuracy_score(cremad_label_ids, preds)
    ax.set_title(f'{title}\n(Acc: {acc:.1%})')

plt.suptitle('CREMA-D Classification with RAVDESS-Trained Methods (Zero-Shot Transfer)', y=1.05)
plt.tight_layout()
plt.savefig(local_output_dir / 'cremad_confusion_matrices.png', dpi=220)
plt.show()

# Per-class F1 bar chart
per_class_rows = []
for method, preds in [('trained_head', trained_preds), ('direction_only', direction_preds)]:
    report = classification_report(cremad_label_ids, preds, target_names=label_names, output_dict=True)
    for name in label_names:
        per_class_rows.append({'method': method, 'emotion': name, 'f1': report[name]['f1-score']})
per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(local_output_dir / 'cremad_per_class_f1.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=per_class_df, x='emotion', y='f1', hue='method', ax=ax)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 on CREMA-D (Zero-Shot Transfer from RAVDESS)')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(local_output_dir / 'cremad_per_class_f1.png', dpi=220)
plt.show()

## UMAP: RAVDESS vs CREMA-D Embeddings

Visualize both datasets in the same embedding space to see if emotion clusters align.

In [ ]:
try:
    import umap

    # Combine RAVDESS test + CREMA-D embeddings
    ravdess_test_emb = ravdess_embeddings[ravdess_test_mask]
    combined = np.vstack([ravdess_test_emb, cremad_embeddings])
    combined_labels = np.concatenate([ravdess_test_ids, cremad_label_ids])
    combined_dataset = np.array(['RAVDESS'] * len(ravdess_test_emb) + ['CREMA-D'] * len(cremad_embeddings))

    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
    coords = reducer.fit_transform(combined)

    plot_df = pd.DataFrame({
        'umap_1': coords[:, 0], 'umap_2': coords[:, 1],
        'emotion': [label_names[i] for i in combined_labels],
        'dataset': combined_dataset,
    })

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, ds_name in zip(axes, ['RAVDESS', 'CREMA-D']):
        subset = plot_df[plot_df['dataset'] == ds_name]
        sns.scatterplot(data=subset, x='umap_1', y='umap_2', hue='emotion', alpha=0.5, s=25, ax=ax)
        ax.set_title(f'{ds_name} Embeddings')
        ax.legend(fontsize=7)
    plt.suptitle('UMAP: Shared Embedding Space (RAVDESS vs CREMA-D)', y=1.02)
    plt.tight_layout()
    plt.savefig(local_output_dir / 'umap_ravdess_cremad.png', dpi=220)
    plt.show()
except ImportError:
    print('umap-learn not installed — skipping UMAP visualization.')

In [ ]:
# --- Drive backup ---
if not IS_COLAB:
    print('Google Drive sync is intended for Colab runtimes. Skipping.')
elif drive_output_dir is None:
    print('Drive output directory is not configured.')
else:
    import shutil
    if drive_output_dir.exists(): shutil.rmtree(drive_output_dir)
    drive_output_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output_dir, drive_output_dir)
    print('Backed up to:', drive_output_dir)

print('\nOutput files:')
for p in sorted(local_output_dir.iterdir()):
    print(f'  {p.name}')